In [ ]:
!unzip /content/dentix_split.zip

Streaming output truncated to the last 5000 lines.
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5911.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5915.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5923.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5926.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5927.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5930.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5933.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5934.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5939.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5944.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mouth_ulcer_Mouth_Ulcer_0_5945.jpeg  
  inflating: dentix_split/train/mouth_ulcer/mou

In [ ]:
ls dentix_split/

train/  val/


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
data_dir = "/content/dentix_split"

train_dataset = datasets.ImageFolder(
    root=f"{data_dir}/train",
    transform=train_transforms
)

val_dataset = datasets.ImageFolder(
    root=f"{data_dir}/val",
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

class_names, num_classes


(['calculus',
  'caries',
  'gingivitis',
  'hypodontia',
  'mouth_ulcer',
  'tooth_discoloration'],
 6)

In [ ]:
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 104MB/s]


In [ ]:
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [ ]:
def train_model(model, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Loss: {running_loss:.4f} "
              f"Train Acc: {train_acc:.2f} "
              f"Val Acc: {val_acc:.2f}")


In [ ]:
train_model(model, epochs=10)


Epoch [1/10] Loss: 612.7270 Train Acc: 0.63 Val Acc: 0.73
Epoch [2/10] Loss: 461.3890 Train Acc: 0.72 Val Acc: 0.71
Epoch [3/10] Loss: 436.4725 Train Acc: 0.73 Val Acc: 0.72
Epoch [4/10] Loss: 414.7710 Train Acc: 0.74 Val Acc: 0.75
Epoch [5/10] Loss: 411.2797 Train Acc: 0.74 Val Acc: 0.76
Epoch [6/10] Loss: 396.0818 Train Acc: 0.75 Val Acc: 0.77
Epoch [7/10] Loss: 391.0758 Train Acc: 0.75 Val Acc: 0.78
Epoch [8/10] Loss: 394.4386 Train Acc: 0.75 Val Acc: 0.78
Epoch [9/10] Loss: 378.2130 Train Acc: 0.76 Val Acc: 0.79
Epoch [10/10] Loss: 381.5887 Train Acc: 0.76 Val Acc: 0.77


In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "classes": class_names
}, "dentix_resnet18_demo.pkl")


In [ ]:
from PIL import Image

def predict_image(img_path):
    model.eval()
    image = Image.open(img_path).convert("RGB")
    image = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return class_names[pred.item()], conf.item()





In [ ]:
predict_image("/content/dentix_split/val/calculus/calculus_(1).jpg")


('calculus', 0.5415361523628235)

In [ ]:
predict_image("/content/dentix_split/val/caries/caries_100.jpg")


('gingivitis', 0.5711824297904968)

In [ ]:
predict_image("/content/dentix_split/val/gingivitis/gingivitis_(1).jpg")


('gingivitis', 0.620020866394043)

In [ ]:
predict_image("/content/dentix_split/val/hypodontia/hypodontia_(10).JPG")


('hypodontia', 0.999657154083252)

In [ ]:
predict_image("/content/dentix_split/val/mouth_ulcer/mouth_ulcer_102.jpg")


('mouth_ulcer', 0.9904858469963074)

In [ ]:
predict_image("/content/dentix_split/val/tooth_discoloration/tooth_discoloration_10.jpg")


('tooth_discoloration', 0.9627267718315125)

In [ ]:
predict_image("/content/images (2).jpg")


('tooth_discoloration', 0.9252172708511353)

In [ ]:
predict_image("/content/images (3).jpg")


('caries', 0.5457748174667358)

In [ ]:
predict_image("/content/images (4).jpg")


('caries', 0.9117869734764099)